# phoneme-entropy — Colab quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suchirsalhan/phoneme/blob/main/examples/colab_quickstart.ipynb)

[`phoneme-entropy`](https://pypi.org/project/phoneme-entropy/) provides phoneme-level
entropy, informativity, and maximum-entropy tools. It bundles the entropy estimators
from [`entropy-estimators`](https://github.com/fermosc24/entropy-estimators) and adds
phonology-oriented analysis utilities.

This notebook runs end-to-end on Colab using the **bundled CELEX sample** — no data
download or credentials required. A final optional section shows how to load the
full **gated** Hugging Face dataset if you hold a CELEX/LDC licence.

**Pipeline:** install → load data → phoneme frequencies → symmetric-Dirichlet α →
segmental informativity & prefix entropy → maximum-entropy fit.


## 0. Install

Install the package (with the `plot` extra for the figures) straight from PyPI.


In [ ]:
!pip install -q "phoneme-entropy[plot]"


In [ ]:
import phoneme_entropy
print("phoneme-entropy", phoneme_entropy.__version__)


## 1. Load the bundled CELEX sample

`load_sample()` returns a small CELEX-derived English lexicon shipped inside the
package. The `Word` column holds **space-separated phonemes**; `Frequency` holds
corpus frequencies.


In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

from phoneme_entropy.data import load_sample

sample = load_sample()
sample.head()


## 2. Phoneme unigram frequencies


In [ ]:
phonfreqs = defaultdict(int)
for word in sample["Word"]:
    for p in word.split():
        phonfreqs[p] += 1
phonfreqs = pd.DataFrame(phonfreqs.items(), columns=["Phoneme", "Frequency"])
phonfreqs.head()


## 3. Symmetric-Dirichlet α from the phoneme distribution

`estimate_alpha_entropy` inverts an entropy estimate to recover the concentration
parameter α of a symmetric Dirichlet prior. `n_boot=0` gives a deterministic point
estimate; increase it (and seed NumPy) for a bootstrap standard error.


In [ ]:
from phoneme_entropy import estimate_alpha_entropy

alpha, se = estimate_alpha_entropy(phonfreqs["Frequency"], n_boot=0)
print("alpha =", alpha, "| se =", se)


### Predicted vs. observed ranks (Beta order statistics)

`plot_ranks` overlays the observed phoneme probabilities on the predicted
order-statistic curve of the fitted symmetric Dirichlet, with confidence bands.


In [ ]:
import matplotlib.pyplot as plt
from phoneme_entropy.dirichlet import plot_ranks

plot_ranks(phonfreqs["Frequency"], alpha=0.7, types="Reversed", loglog=False)
plt.show()


## 4. Segmental informativity & prefix entropy

- `segment_informativity` — frequency-weighted **surprisal** of the next phoneme.
- `phoneme_prefix_entropy` — mean **entropy** of the words still compatible after
  each phoneme (uses the bundled entropy estimators, e.g. `CWJ`, `ML`, `NSB`).


In [ ]:
from phoneme_entropy import segment_informativity, phoneme_prefix_entropy

surprisal   = segment_informativity(sample["Word"])
prefix_H    = phoneme_prefix_entropy(sample["Word"])
surprisal.head()


## 5. Maximum-entropy fit

Form frequency-weighted target expectations of `(Surprisal, Entropy)` and fit the
maximum-entropy distribution whose feature expectations match them.


In [ ]:
from phoneme_entropy import compute_maxent_from_matrix

merged = phonfreqs.merge(surprisal, on="Phoneme").merge(prefix_H, on="Phoneme")
p = merged["Frequency"] / merged["Frequency"].sum()
targets = np.array([np.sum(p * merged["Surprisal"]), np.sum(p * merged["Entropy"])])
F = merged[["Surprisal", "Entropy"]].to_numpy()

probs, lambdas = compute_maxent_from_matrix(F, targets)
print("Lagrange multipliers (lambda):", lambdas)


In [ ]:
import seaborn as sns
sns.regplot(x=np.log(p), y=np.log(probs))
plt.xlabel("(log) observed probability")
plt.ylabel("(log) predicted probability")
plt.show()


## 6. (Optional) Load the full gated dataset

The complete CELEX-derived sample is mirrored as a **gated, permissions-only**
Hugging Face dataset
[`suchirsalhan/celex-en-phoneme-sample`](https://huggingface.co/datasets/suchirsalhan/celex-en-phoneme-sample).
Access is granted to holders of a valid CELEX/LDC licence
([LDC96L14](https://catalog.ldc.upenn.edu/docs/LDC96L14/celex.readme.html)).

Once your access request is approved, log in and load it:


In [ ]:
# Uncomment to run (needs an approved access request + HF token):
# !pip install -q "phoneme-entropy[data]"
# from huggingface_hub import notebook_login; notebook_login()
# from phoneme_entropy.data import load_hf_dataset
# ds = load_hf_dataset(split="train")
# print(ds)


---

**Learn more:** [PyPI](https://pypi.org/project/phoneme-entropy/) ·
[GitHub](https://github.com/suchirsalhan/phoneme) ·
[upstream entropy-estimators](https://github.com/fermosc24/entropy-estimators).
Licensed MIT; please cite via `CITATION.cff`.
